# 4. Production feature-distribution diagnostics

Inspect the all-comment primary predictors and optional root appendix predictors before fitting stages 7 and 8. Full-data DuckDB scans test completeness, ranges, and basic invariants; deterministic bounded samples support plots, correlations, and selected-versus-unselected comparisons without loading either complete choice set into memory.

Success means: no missing or non-finite model values, no invalid bounded values, non-constant predictors, plausible distribution shapes, and no unexplained near-duplicate predictors.

In [ ]:
from __future__ import annotations

import json
import math
import os
from pathlib import Path

import duckdb
import numpy as np
import pandas as pd
import pyarrow.parquet as pq
import seaborn as sns
from IPython.display import display
from commentgap_analysis.feature_diagnostics import (
    BINARY_FEATURES,
    feature_group,
    full_distribution_summary,
    invariant_checks,
    ks_statistic,
    plot_aqua_expected_distributions,
    plot_distribution_grid,
    plot_selection_contrasts,
    plot_spearman_correlation_heatmap,
    quote_identifier,
    save_figure,
    within_story_variation,
)

SEED = int(os.getenv("COMMENTGAP_DIAGNOSTIC_SEED", "20260824"))
SAMPLE_ROWS = int(os.getenv("COMMENTGAP_DIAGNOSTIC_SAMPLE_ROWS", "100000"))
CORRELATION_ROWS = int(os.getenv("COMMENTGAP_DIAGNOSTIC_CORRELATION_ROWS", "50000"))
DUCKDB_THREADS = int(os.getenv("COMMENTGAP_DIAGNOSTIC_THREADS", "8"))
STRICT_CHECKS = os.getenv("COMMENTGAP_DIAGNOSTIC_STRICT", "1") == "1"
SOURCE_FEATURE_ROOT = Path(os.getenv("COMMENTGAP_FEATURE_ROOT", "model_output/selection_2025/features"))
MODEL_DATA_ROOT = Path(os.getenv("COMMENTGAP_MODEL_DATA_ROOT", "model_output/selection_2025/model_data"))
OUTPUT_ROOT = Path(
    os.getenv(
        "COMMENTGAP_FEATURE_DIAGNOSTIC_ROOT",
        "model_output/selection_2025/feature_diagnostics",
    )
)
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
sns.set_theme(style="whitegrid", context="notebook")
connection = duckdb.connect()
connection.execute(f"SET threads = {DUCKDB_THREADS}")
{"sample_rows": SAMPLE_ROWS, "correlation_rows": CORRELATION_ROWS, "output": str(OUTPUT_ROOT)}

## 1. Load and validate the production feature contract

This fails early if feature assembly has not been rerun with manifest v4 and the 20 expected AQuA dimensions. The composite AQuA scores must remain outside the primary feature lists.

In [ ]:
build_state = json.loads((SOURCE_FEATURE_ROOT / "build_state.json").read_text())
provenance = json.loads((MODEL_DATA_ROOT / "provenance_manifest.json").read_text())
registry = json.loads((MODEL_DATA_ROOT / "feature_manifest.json").read_text())
assert build_state["status"] == "complete"
assert provenance["watermark"] == "INFERENCE"
assert provenance["models"]["aqua"]["watermark"] == "PRODUCTION"
assert registry["version"] >= 4, "Re-run commentgap-features to create the AQuA primary-model contract"

choice_paths = {scope: MODEL_DATA_ROOT / f"choice_set_{scope}.parquet" for scope in ("root", "all")}
model_features = {scope: list(registry["models"][scope]["features"]) for scope in choice_paths}
for scope, path in choice_paths.items():
    assert path.exists(), path
    assert len(model_features[scope]) == (39 if scope == "root" else 41)
    assert len(model_features[scope]) == len(set(model_features[scope]))
    aqua = [name for name in model_features[scope] if name.startswith("aqua_")]
    assert len(aqua) == 20 and all(name.endswith("_expected") for name in aqua)
    assert "aqua_score_expected" not in model_features[scope]
    available = set(pq.ParquetFile(path).schema_arrow.names)
    missing = set(model_features[scope]) - available
    assert not missing, (scope, sorted(missing))

contract = pd.DataFrame(
    [
        {
            "scope": scope,
            "rows": pq.ParquetFile(path).metadata.num_rows,
            "model_features": len(model_features[scope]),
            "aqua_expected_features": sum(name.startswith("aqua_") for name in model_features[scope]),
        }
        for scope, path in choice_paths.items()
    ]
)
display(contract)

## 2. Full-data distribution summaries

One aggregate query per scope scans every candidate row. Approximate quantiles are sufficient for diagnostics and avoid retaining raw values. Range failures, missing values, non-finite values, and constant predictors are treated as critical. Heavy-tail flags are informational because several count-derived variables are intentionally right-skewed even after `log1p`.

In [ ]:
distribution_summary = pd.concat(
    [
        full_distribution_summary(
            scope,
            connection=connection,
            choice_paths=choice_paths,
            model_features=model_features,
            registry=registry,
        )
        for scope in ("root", "all")
    ],
    ignore_index=True,
)
distribution_summary.to_csv(OUTPUT_ROOT / "full_distribution_summary.csv", index=False)
flags = distribution_summary[
    (distribution_summary["nulls"] > 0)
    | (distribution_summary["nonfinite"] > 0)
    | distribution_summary["range_violation"]
    | distribution_summary["constant_or_near_constant"]
    | distribution_summary["heavy_right_tail"]
]
display(flags if len(flags) else pd.DataFrame({"status": ["No distribution flags"]}))

## 3. Full-data row-level invariants

These checks target relationships that marginal summaries cannot detect: unique candidate keys, informative choice sets, sentiment components summing to one, maximum toxicity not falling below mean toxicity, and valid binary values.

In [ ]:
invariants = pd.concat(
    [
        invariant_checks(
            scope,
            connection=connection,
            choice_paths=choice_paths,
            model_features=model_features,
        )
        for scope in ("root", "all")
    ],
    ignore_index=True,
)
invariants.to_csv(OUTPUT_ROOT / "row_invariant_checks.csv", index=False)
display(invariants)

within_story = pd.concat(
    [
        within_story_variation(
            scope,
            connection=connection,
            choice_paths=choice_paths,
            model_features=model_features,
        )
        for scope in ("root", "all")
    ],
    ignore_index=True,
)
within_story.to_csv(OUTPUT_ROOT / "within_story_feature_variation.csv", index=False)
display(within_story.sort_values(["scope", "proportion_stories_with_variation"]).groupby("scope").head(12))

## 4. Deterministic bounded samples

Reservoir sampling is reproducible and row-bounded. It is used only for visualization and effect-size diagnostics; the preceding summaries and invariants use every row.

In [ ]:
samples = {}
identifier_columns = [
    "story_id", "comment_id", "n_candidates", "n_picks",
    "curator_selected", "audience_selected_draw_01",
]
for scope, path in choice_paths.items():
    available = set(pq.ParquetFile(path).schema_arrow.names)
    extras = [
        name for name in ("sentiment_neutral", "toxicity_mean_probability", "aqua_score_expected")
        if name in available
    ]
    columns = list(dict.fromkeys(identifier_columns + model_features[scope] + extras))
    select = ", ".join(quote_identifier(name) for name in columns)
    query = (
        f"SELECT {select} FROM read_parquet(?) "
        f"USING SAMPLE reservoir({SAMPLE_ROWS} ROWS) REPEATABLE ({SEED})"
    )
    samples[scope] = connection.execute(query, [str(path)]).fetchdf()
sample_accounting = pd.DataFrame(
    [{"scope": scope, "sample_rows": len(frame), "sample_stories": frame["story_id"].nunique()} for scope, frame in samples.items()]
)
display(sample_accounting)

## 5. Distribution plots

Plots are clipped only for display at the sampled 0.5th and 99.5th percentiles; the axis annotation reports that clipping. Binary predictors are shown without clipping. Root and all-comment AQuA distributions are overlaid on their native 0–3 scale.

In [ ]:
for scope in ("root", "all"):
    for group in ("text_nlp", "semantic_timing_activity", "author_history", "reply_structure"):
        selected = [name for name in model_features[scope] if feature_group(name, model_features) == group]
        if selected:
            plot_distribution_grid(
                scope,
                selected,
                group,
                samples=samples,
                registry=registry,
                output_root=OUTPUT_ROOT,
            )

In [ ]:
aqua_features = [name for name in model_features["root"] if feature_group(name, model_features) == "aqua_expected"]
plot_aqua_expected_distributions(samples, aqua_features, registry, OUTPUT_ROOT)

## 6. Correlation and redundancy checks

Spearman correlations are calculated on at most 50,000 sampled rows per scope. Pairs with absolute correlation at least 0.90 are listed for investigation; a high correlation is not automatically an error, particularly among related AQuA dimensions and discussion-activity measures.

In [ ]:
high_correlation_rows = []
for scope in ("root", "all"):
    frame = samples[scope][model_features[scope]].head(CORRELATION_ROWS).apply(pd.to_numeric, errors="coerce")
    correlation = frame.corr(method="spearman")
    correlation.to_csv(OUTPUT_ROOT / f"{scope}_spearman_correlations.csv")
    plot_spearman_correlation_heatmap(correlation, scope, OUTPUT_ROOT)
    for left_index, left in enumerate(correlation.columns):
        for right in correlation.columns[left_index + 1:]:
            value = correlation.loc[left, right]
            if np.isfinite(value) and abs(value) >= 0.90:
                high_correlation_rows.append({"scope": scope, "feature_1": left, "feature_2": right, "spearman": value})
high_correlations = pd.DataFrame(high_correlation_rows).sort_values("spearman", key=abs, ascending=False) if high_correlation_rows else pd.DataFrame(columns=["scope", "feature_1", "feature_2", "spearman"])
high_correlations.to_csv(OUTPUT_ROOT / "high_correlation_pairs.csv", index=False)
display(high_correlations.head(50))

## 7. Selected-versus-unselected distribution tests

Standardized mean differences (SMD) and two-sample Kolmogorov–Smirnov statistics summarize separation for curator and audience selections. They are descriptive checks, not causal estimates or confirmatory hypothesis tests; no p-values are reported because the very large sample would make trivial differences statistically significant.

In [ ]:
contrast_rows = []
for scope, frame in samples.items():
    for selector, outcome in (("curator", "curator_selected"), ("audience", "audience_selected_draw_01")):
        selected_mask = frame[outcome].astype(bool).to_numpy()
        for feature in model_features[scope]:
            values = pd.to_numeric(frame[feature], errors="coerce").to_numpy(float)
            selected = values[selected_mask]
            unselected = values[~selected_mask]
            pooled_sd = math.sqrt((np.nanvar(selected, ddof=1) + np.nanvar(unselected, ddof=1)) / 2)
            smd = (np.nanmean(selected) - np.nanmean(unselected)) / pooled_sd if pooled_sd > 0 else np.nan
            contrast_rows.append(
                {
                    "scope": scope, "selector": selector, "feature": feature,
                    "selected_mean": np.nanmean(selected), "unselected_mean": np.nanmean(unselected),
                    "standardized_mean_difference": smd,
                    "ks_statistic": ks_statistic(selected, unselected),
                }
            )
selection_contrasts = pd.DataFrame(contrast_rows)
selection_contrasts.to_csv(OUTPUT_ROOT / "selection_distribution_contrasts.csv", index=False)

for scope in ("root", "all"):
    subset = selection_contrasts[selection_contrasts["scope"] == scope].copy()
    plot_selection_contrasts(selection_contrasts, scope, registry, OUTPUT_ROOT)
display(selection_contrasts.sort_values("ks_statistic", ascending=False).head(30))

## 8. Final diagnostic status

Critical failures stop strict runs here, after tables and plots have been written. High correlations and heavy-tail warnings remain review items rather than automatic failures. Set `COMMENTGAP_DIAGNOSTIC_STRICT=0` to inspect a known-problematic build without raising the final assertion.

In [ ]:
critical_distribution_failures = distribution_summary[
    (distribution_summary["nulls"] > 0)
    | (distribution_summary["nonfinite"] > 0)
    | distribution_summary["range_violation"]
    | distribution_summary["constant_or_near_constant"]
]
critical_invariant_failures = invariants[invariants["failures"] > 0]
unidentified_features = within_story[within_story["stories_with_variation"] == 0]
status = {
    "status": "PASS" if critical_distribution_failures.empty and critical_invariant_failures.empty and unidentified_features.empty else "REVIEW_REQUIRED",
    "critical_distribution_failures": len(critical_distribution_failures),
    "critical_invariant_failures": len(critical_invariant_failures),
    "features_without_within_story_variation": len(unidentified_features),
    "informational_heavy_tail_flags": int(distribution_summary["heavy_right_tail"].sum()),
    "high_correlation_pairs": len(high_correlations),
    "output_root": str(OUTPUT_ROOT),
}
(OUTPUT_ROOT / "diagnostic_status.json").write_text(json.dumps(status, indent=2, sort_keys=True) + "\n")
display(status)
if STRICT_CHECKS:
    assert status["status"] == "PASS" and unidentified_features.empty, {
        "distribution": critical_distribution_failures[["scope", "feature", "nulls", "nonfinite", "range_violation", "constant_or_near_constant"]].to_dict("records"),
        "invariants": critical_invariant_failures.to_dict("records"),
        "features_without_within_story_variation": unidentified_features.to_dict("records"),
    }
status